In [ ]:
### Notebook global parameters ###

# Performance related
BATCH_SIZE = 10_000 # Used in insert_data() 

In [ ]:
### Imports ###
import os
import sqlite3, psycopg, dotenv

_ = dotenv.load_dotenv()

In [ ]:
### Methods and functions ###

def load_tables_names(conn: sqlite3.Connection) -> list[str]:
    """
    From a SQLite connection, load all available tables.
    To do that, look at tables in table `sqlite_master`.
    Returns all data table names in a list of strings
    """
    # Query the database
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type = 'table' AND name NOT LIKE 'sqlite_%' ORDER BY name;")
    rows = cursor.fetchall()

    # Format
    table_names = [row[0] for row in rows]

    # Finalize
    cursor.close()
    return table_names


def clean_integer_columns(conn: sqlite3.Connection, table_name: str) -> None:
    """Clean wrongly formatted values in SQLite integer columns in batches. Update source database"""

    def is_column_type(column_type: str) -> bool:
        # Prepare verification
        column_type = column_type.upper() 
        # Number types in SQLite to be associated with actual NUMERIC in PGSQL
        nb_types = ['INT', 'NUMERIC', 'NUM']
        for type in nb_types:
            if type in column_type:
                return True
        return False


    # Cursor used for the function
    cursor = conn.cursor()

    # Identify columns declared as integers
    cursor.execute(f"PRAGMA table_info({table_name})")
    integer_columns = [row[1] for row in cursor.fetchall() if is_column_type(row[2])]

    # Process each integer column independently
    for column in integer_columns:
        offset = 0

        while True:
            # Fetch one batch of values, exit if empty
            cursor.execute(f'SELECT rowid, "{column}" FROM "{table_name}" LIMIT {BATCH_SIZE} OFFSET {offset}')
            batch = cursor.fetchall()
            if not batch: break

            updates = []
            for rowid, value in batch:
                # No need to correct anything if it is a string (i.e. not a number)
                if not isinstance(value, str): continue

                # Remove thousands separators
                cleaned = value.replace(",", "").replace(".", "").strip()

                try:
                    cleaned = int(cleaned)
                except ValueError:
                    cleaned = None

                # Save modification to update list
                updates.append((cleaned, rowid))

            # Update only values that were originally strings
            cursor.executemany(f'UPDATE "{table_name}" SET "{column}" = ? WHERE rowid = ?', updates)

            offset += len(batch)

        print(f"> Cleaned column {column}")

    if integer_columns: # i.e. if there was at least one
        conn.commit()
    cursor.close()
    print(f"> Table {table_name} cleaned")


def create_table_in_destination(conn_from: sqlite3.Connection, table_name: str, conn_to: psycopg.Connection, schema_name: str) -> None:
    """
    From a table name, fetch information in source database, 
    to create the equivalent in the destination database.
    Handles primary keys.
    Attention, Destination database requires a schema name!
    """

    # Fetch information from source database
    cursor_from = conn_from.cursor()
    cursor_from.execute(f"PRAGMA table_info({table_name})") # Gives information about columns: (cid, name, type, notnull, default_value, pk)
    rows = cursor_from.fetchall() 

    # Prepare the query to execute in Destination database
    sql = '\n'
    sql += f'CREATE TABLE {schema_name}.{table_name} (\n'
    for column in rows: 
        if column[5] == 1: # i.e. primary key
            sql += f'    {column[1]} SERIAL PRIMARY KEY,\n'
        elif column[2] == "NUM":
            sql += f'    {column[1]} NUMERIC,\n'
        else: # i.e. not a primary key
            sql += f'    {column[1]} {column[2]},\n'
    sql = sql[:sql.rfind(',')] + '\n'
    sql += f')\n'
    
    # Create the table in the destination database
    cursor_to = conn_to.cursor()
    cursor_to.execute(sql)

    # Finalize
    cursor_to.close()
    cursor_from.close()
    print(f'> Table {schema_name}.{table_name} created.')


def create_foreign_keys_in_destination(conn_from: sqlite3.Connection, conn_to: psycopg.Connection, schema_name: str, table_name: str) -> None:
    """
    With all table names given as parameter, can check each table for foreign keys.
    If some are found, they are simply reproduced in destination database.
    Attention A foreign keys link two tables, this function suppose tables already exists.
    """

    # Prepare before loop
    sql = ""
    cursor_from = conn_from.cursor()
    cursor_from.execute(f"PRAGMA foreign_key_list({table_name})") # Gives information about FK: (id, seq, table, from, to, on_update, on_delete, match)
    rows = cursor_from.fetchall()

    # Prepare SQL for all retrieved foreign keys
    for row in rows:
        sql += f"ALTER TABLE {table_name} ADD CONSTRAINT fk_{table_name}_{row[2]} FOREIGN KEY ({row[3]}) REFERENCES {schema_name}.{row[2]}({row[4]});\n"

    # Create all FKs
    cursor_to = conn_to.cursor()
    cursor_to.execute(sql)

    # Finalize
    cursor_to.close()
    cursor_from.close()
    print(f'> Foreign keys of table {schema_name}.{table_name} created.')


def insert_data(conn_from: sqlite3.Connection, conn_to: psycopg.Connection, schema_name: str, table_name: str) -> None:
    """
    Insert data into destination database-schema-table, from source database-table
    """

    # Prepare
    cursor_from = conn_from.cursor()
    cursor_to = conn_to.cursor()

    # Make the server execute the query
    cursor_from.execute(f"SELECT * FROM {table_name}")

    # Parse meta information
    columns = [col[0] for col in cursor_from.description]

    # While we have rows, continue
    while True:

        # Fetch the next rows
        batch = cursor_from.fetchmany(BATCH_SIZE)
        if not batch: break # If no more, exit loop

        # Prepare query components
        placeholders = ", ".join(["%s"] * len(columns))
        column_names = ", ".join(columns)

        # Build the query
        query = f"INSERT INTO {schema_name}.{table_name} ({column_names}) VALUES ({placeholders})"

        # Insert batched rows into the destination database
        cursor_to.executemany(query, batch)

        print(f"> Inserted {len(batch)} rows")

    # Finalize
    cursor_from.close()
    cursor_to.close()
    print(f"> Table {schema_name}.{table_name} has data inserted")

In [ ]:
# Databases Connections
conn_from = sqlite3.connect(os.getenv("SQLITE_PATH"), autocommit=True)
conn_to = psycopg.connect(
    host=os.getenv("PGHOST"),
    port=os.getenv("PGPORT"),
    dbname=os.getenv("PGDATABASE"),
    user=os.getenv("PGUSER"),
    password=os.getenv("PGPASSWORD"),
)

try:
    # List all tables to be migrated
    tables_names = load_tables_names(conn_from)

    # Loop to create the tables, and fill them with data
    for table_name in tables_names:
        clean_integer_columns(conn_from, table_name)
        create_table_in_destination(conn_from, table_name, conn_to, 'public')
        insert_data(conn_from, conn_to, 'public', table_name)

    # Loop to add foreign keys. This not done in previous loop to be sure that referenced table-columns already exists
    for table_name in tables_names:
        create_foreign_keys_in_destination(conn_from, conn_to, 'public', table_name)

    # Validate everything
    conn_to.commit()

# If something goes wrong, rollback
except Exception:
    conn_to.rollback()
    raise

# Finalize
finally:
    conn_from.close()
    conn_to.close()